# Island Hopper — Analysis Notebook

Analysis of the Island Hopper routing problem: can a small aircraft fly from Milan (MXP) to Sydney (SYD)? Run the cells in order.

Covered:
1. Data validation
2. Sydney reachability: minimum-stops route (BFS)
3. Fastest route (Dijkstra, with refuelling and customs)
4. Reachability at 4,000 km
5. Farthest reachable airport from Milan
6. Range sensitivity analysis — 1,500–8,000 km
7. Visualisations


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd

from src.data_loader import load_airports
from src.routing import (
    find_route_bfs,
    find_route_dijkstra,
    reachable_within,
)
from src.analysis import (
    find_farthest_reachable_airport,
    run_range_sensitivity_analysis,
)
from src import visualization


## 1. Load and validate the airport data

In [ ]:
airports = load_airports()

print("Columns:", airports.columns.tolist())
print("Rows:", len(airports))
print("Airport types:", airports["type"].value_counts().to_dict())
print(
    "Missing coordinates:",
    int(airports["latitude"].isna().sum() + airports["longitude"].isna().sum()),
)
print("Duplicate airport IDs:", int(airports["id"].duplicated().sum()))


## 2. Sydney reachability — minimum number of flight legs (BFS)

In [ ]:
bfs_route = find_route_bfs(airports, "MXP", "SYD", max_range_km=4000, max_stops=10)

if bfs_route is None:
    print("Sydney is not reachable within the stop limit.")
else:
    print("Route:", " -> ".join(bfs_route))
    print("Flight legs:", len(bfs_route) - 1)
    print("Intermediate stops:", len(bfs_route) - 2)


## 3. Fastest route (Dijkstra) — flight time, refuelling and customs

In [ ]:
fastest_route, total_time_h = find_route_dijkstra(
    airports,
    "MXP",
    "SYD",
    max_range_km=4000,
    speed_kmh=500,
    refuel_hours=1,
    customs_hours=2,
    max_stops=10,
)

if fastest_route is None:
    print("Sydney is not reachable.")
else:
    print("Route:", " -> ".join(fastest_route))
    print("Flight legs:", len(fastest_route) - 1)
    print("Intermediate stops:", len(fastest_route) - 2)
    print("Fastest travel time:", round(total_time_h, 2), "hours")


## 4. Reachability at 4,000 km

In [ ]:
reachable = reachable_within(airports, "MXP", max_range_km=4000, max_stops=10)
reachable.pop("MXP", None)

print("Number of reachable airports:", len(reachable))

max_legs = max(reachable.values())
print("Maximum flight legs:", max_legs)
print("Maximum intermediate stops:", max_legs - 1)
print("Airports requiring the most flight legs:")
for label in sorted(c for c, legs in reachable.items() if legs == max_legs):
    print(" -", label)


## 5. Farthest reachable airport from Milan

In [ ]:
farthest = find_farthest_reachable_airport(
    airports, "MXP", max_range_km=4000, max_stops=10
)

if farthest is None:
    print("No reachable airports.")
else:
    for key in [
        "name", "iata_code", "ident",
        "distance_km", "flight_legs", "intermediate_stops",
    ]:
        print(f"{key.replace('_', ' ').title()}:", farthest[key])


## 6. Range sensitivity analysis — 1,500–8,000 km

In [ ]:
ranges = [1500, 2000, 2500, 3000, 3500, 4000, 5000, 6000, 7000, 8000]
sensitivity = run_range_sensitivity_analysis(
    airports,
    origin_code="MXP",
    destination_code="SYD",
    ranges=ranges,
    speed_kmh=500,
    refuel_hours=1,
    customs_hours=2,
    max_stops=10,
)
sensitivity


## 7. Visualisations

In [ ]:
if bfs_route is not None:
    fig = visualization.plot_route_map(
        airports,
        bfs_route,
        title="MXP → SYD minimum-stops route (4,000 km range)",
    )
    plt.show()


In [ ]:
fig = visualization.plot_reachability_vs_range(sensitivity)
plt.show()


In [ ]:
fig = visualization.plot_min_stops_vs_range(sensitivity)
plt.show()


In [ ]:
fig = visualization.plot_fastest_time_vs_range(sensitivity)
plt.show()


In [ ]:
farthest_label = (
    farthest["iata_code"] or farthest["ident"] or farthest["name"]
)
fig = visualization.plot_farthest_airport_map(
    airports,
    origin_label="MXP",
    farthest_label=farthest_label,
    title=f"Farthest reachable airport from MXP: {farthest_label}",
)
plt.show()


## 8. Conclusions

- **Sydney is reachable** from Milan Malpensa (MXP) with a 4,000 km range: the minimum-stops route has **5 flight legs (4 intermediate stops)**.
- The **fastest route** takes about **43.5 hours**: `MXP -> LWN -> NAG -> VO94 -> KTA -> SYD`.
- At the 4,000 km range, **5,280 airports** are reachable from Milan.
- The **farthest reachable airport** is **CHT** (Inia William Tuuta Memorial Airport), about **19,558 km** from Milan, reachable with 5 flight legs.
- Comparing ranges from 1,500 km to 8,000 km: Sydney is not reachable at 1,500 km; it becomes reachable from 2,000 km onward, and the minimum number of stops drops from 8 to 2 as the range grows.

Reminder: the speed, range, refuelling and customs assumptions strongly shape these numbers — they have not been verified against real flight data.
